**Import tableau final**

In [5]:
!pip install --upgrade google-cloud-bigquery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client(project="projet-riskhorizon-2276")


In [15]:
query = """
SELECT *
FROM `projet-riskhorizon-2276.3_Mart.tableau_final_table`
"""

df = client.query(query).to_dataframe()

df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,dette_totale_ext,ratio_dette_credit_ext,nb_credits_en_retard_ext,retard_max_ext,montant_total_retard_ext,nb_prolongations_ext,nb_types_credit_ext,nb_demandes_passees_int,nb_refus_passes_int,montant_total_emprunte_passe_int
0,355557,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4450.5,...,122050.98,0.474226,0,0,0.0,0,2,1,0,75312.0
1,211865,0,CASH_LOANS,F,False,True,0,27000.0,45000.0,4383.0,...,637623.00,0.421575,0,0,0.0,0,2,6,0,660501.0
2,364118,0,CASH_LOANS,F,False,True,0,29250.0,45000.0,4855.5,...,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,5,2,207724.5
3,143626,1,CASH_LOANS,F,False,True,0,31500.0,45000.0,5076.0,...,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,1,0,32503.5
4,372569,0,CASH_LOANS,F,False,False,0,31500.0,45000.0,4450.5,...,29817.00,0.047083,0,0,0.0,0,2,2,0,207616.5


**WORKFLOW COMPLET**

In [7]:
# [ ÉTAPE 1 : COLLECTE & CHARGEMENT ]
#    df = pd.read_csv(...) : Import de vos données brutes (TARGET, SK_ID_CURR, variables).
#         │
# [ ÉTAPE 2 : SÉPARATION DES CIBLES & IDENTIFIANTS ]
#    Extraction de 'y' (TARGET) et 'client_id' (SK_ID_CURR).
#    Création de 'X' en supprimant ces deux colonnes du dataset de travail.
#         │
# [ ÉTAPE 3 : ENCODAGE DES VARIABLES CATÉGORIELLES ]
#    X_encoded = pd.get_dummies(X, drop_first=True) : Transformation du texte en chiffres (0 ou 1).
#    Sauvegarde de 'feature_names' pour l'analyse finale.
#         │
# [ ÉTAPE 4 : LE SPLIT INITIAL (Le Coffre-Fort) ]
#    train_test_split(..., test_size=0.2, stratify=y) : On isole 20% des données (X_test, y_test).
#    Elles restent totalement secrètes et intouchées jusqu'à la fin.
#         │
# [ ÉTAPE 5 : DÉFINITION DU SQUELETTE (Le Pipeline) ]
#    Déclaration de la chaîne de montage sans données :
#      SimpleImputer (médiane) ➔ StandardScaler ➔ LogisticRegression(class_weight="balanced").
#         │
# [ ÉTAPE 6 : VALIDATION CROISÉE (L'Évaluation de la stratégie) ]
#    cross_val_score(model_pipeline, X_train, y_train, cv=5) :
#      Simulation de 5 entraînements / tests internes pour calculer l'AUC Moyen et l'Écart-type (.std()).
#    Objectif : Valider que le modèle est stable et ne fait pas d'overfitting.
#         │
# [ ÉTAPE 7 : LE FIT FINAL (L'Entraînement définitif) ]
#    model_pipeline.fit(X_train, y_train) :
#      Le pipeline apprend DEFINITIVEMENT les médianes, les moyennes et les coefficients sur 100% du train.
#         │
# [ ÉTAPE 8 : LE TEST RÉEL (L'Évaluation finale) ]
#    y_pred_test = model_pipeline.predict_proba(X_test)[:, 1]
#    Calcul de l'AUC Test Final : On vérifie si la performance se maintient sur les données secrètes.
#         │
# [ ÉTAPE 9 : EXTRACTION DE L'IMPORTANCE DES VARIABLES ]
#    coefficients = model_pipeline.named_steps['classifier'].coef_[0]
#    Analyse et tri des variables les plus influentes via le graphique Plotly.
#         │
# [ ÉTAPE 10 : Matrice de confusion ]
#    cm = confusion_matrix(y_test, y_pred_class)
#    cm_normalized = confusion_matrix(y_test, y_pred_class, normalize='true') # Taux par ligne
#    Graphique plotly


**ETAPE 1 à 8**

In [8]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_auc_score, confusion_matrix

#Préparation des données
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])

#Encodage (get_dummies) avant le split
X_encoded = pd.get_dummies(X, drop_first=True)

# Sauvegarde des noms de variables pour l'interprétation
feature_names = X_encoded.columns

# Split initial
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

#Création du Pipeline (preparation usinage)
preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),#=> remplace valeur manquante par mediane
    ('scaler', StandardScaler())# => met à l'échelle toutes les données
])

model_pipeline = Pipeline([
    ('preprocessor', preprocessing),#=> valeurs manquantes + standardisation
    ('classifier', LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))#=> algorithme de Régression Logistique
])

#class_weight="balanced"=>permet d'équilibrer la répartion des target (trop peu de 1)
# TARGET
# 0   91.9268
# 1    8.0732


#Validation Croisée (StratifiedKFold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) #=> n_splits=5 : doit découper les données en 5 morceaux (les folds) pour
cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print("VALIDATION CROISÉE")
print(f"Scores AUC par fold : {cv_scores}")
print(f"Score AUC Moyen : {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print("-" * 30)

# Entraînement Final & Évaluation
model_pipeline.fit(X_train, y_train)

y_pred_test = model_pipeline.predict_proba(X_test)[:, 1]
print(f"AUC Test Final : {roc_auc_score(y_test, y_pred_test):.4f}\n")



VALIDATION CROISÉE
Scores AUC par fold : [0.75034462 0.75046092 0.7545729  0.74570283 0.74999007]
Score AUC Moyen : 0.7502 (+/- 0.0056)
------------------------------
AUC Test Final : 0.7467



In [9]:
print(y.value_counts(normalize=True) * 100)

TARGET
0    91.926751
1     8.073249
Name: proportion, dtype: Float64


**ETAPE 9**

In [10]:
# On récupère les coefficients du modèle linéaire
coefficients = model_pipeline.named_steps['classifier'].coef_[0]

# Création d'un DataFrame pour analyser les poids
importance_df = pd.DataFrame({
    'Variable': feature_names,
    'Coefficient': coefficients,
    'Abs_Coefficient': np.abs(coefficients) # Pour trier par impact absolu
}).sort_values(by='Abs_Coefficient', ascending=False)

# Visualisation des 20 variables les plus influentes
fig_importance = px.bar(
    importance_df.head(20),
    x='Coefficient',
    y='Variable',
    orientation='h',
    title='Top 20 des variables les plus influentes (Coefficients Régression Logistique)',
    labels={'Coefficient': 'Poids du coefficient (Positif = Risque de défaut, Négatif = Sûr)'},
    color='Coefficient',
    color_continuous_scale=px.colors.diverging.RdYlGn[::-1] # Rouge pour risque, Vert pour sûr
)
fig_importance.update_layout(yaxis={'categoryorder':'total ascending'})
fig_importance.show()

In [11]:
new_importance_df = importance_df.copy()
# limiter 4 decimals après virgule
new_importance_df['Coefficient'] = new_importance_df['Coefficient'].apply(lambda x: '{:,.4f}'.format(x))
new_importance_df['Abs_Coefficient'] = new_importance_df['Abs_Coefficient'].apply(lambda x: '{:,.4f}'.format(x))

print(new_importance_df.to_string(index=False))

                                         Variable Coefficient Abs_Coefficient
                                   nb_credits_ext     -1.1698          1.1698
                          nb_credits_clotures_ext      0.8370          0.8370
                            nb_credits_actifs_ext      0.6065          0.6065
                                     EXT_SOURCE_2     -0.4208          0.4208
                                     EXT_SOURCE_3     -0.4162          0.4162
                              nb_refus_passes_int      0.3028          0.3028
                          nb_demandes_passees_int     -0.2875          0.2875
NAME_EDUCATION_TYPE_SECONDARY_/_SECONDARY_SPECIAL      0.1967          0.1967
                                     EXT_SOURCE_1     -0.1897          0.1897
                                    CODE_GENDER_M      0.1691          0.1691
                                   YEARS_EMPLOYED     -0.1556          0.1556
                 montant_total_emprunte_passe_int      0.1471   

In [12]:
from google.cloud import bigquery

# Define your BigQuery project, dataset, and table name
project_id = "projet-riskhorizon-2276"
dataset_id = "3_Mart"
table_id = "coeff_regression_variables"

destination_table = f"{project_id}.{dataset_id}.{table_id}"

# Export df_final to BigQuery
new_importance_df.to_gbq(
    destination_table,
    project_id=project_id,
    if_exists='replace'
)

print(f"DataFrame 'df_final' successfully exported to BigQuery table: {destination_table}")

/tmp/ipykernel_11799/2106634183.py:11: FutureWarning:

to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq

100%|██████████| 1/1 [00:00<00:00, 7639.90it/s]

DataFrame 'df_final' successfully exported to BigQuery table: projet-riskhorizon-2276.3_Mart.coeff_regression_variables


**Etape 10 : Matrice de confusion**

In [13]:
# conversion des probabilités en classes (Seuil par défaut à 0.5)
y_pred_class = (y_pred_test >= 0.5).astype(int)

# calcul de la matrice brute et normalisée
cm = confusion_matrix(y_test, y_pred_class)
cm_normalized = confusion_matrix(y_test, y_pred_class, normalize='true') # Taux par ligne

# création du graphique Plotly
labels_etats = ['Non-Défaut (0)', 'Défaut (1)']

fig_cm = px.imshow(
    cm_normalized,
    text_auto=".2%", # Affiche les taux en pourcentage
    x=labels_etats,
    y=labels_etats,
    labels=dict(x="État PRÉDIT par le modèle", y="État RÉEL du client", color="Taux"),
    title="Matrice de Confusion (Transition Réel vs Prédit)",
    color_continuous_scale="Blues"
)

fig_cm.show()